# Real-time Object Detection + LCOD Benchmark — Ready-to-Play UI

Giữ nguyên workflow **Run All → preflight → UI → Model + Input + Prompt → Play**.

**Closed-set:** YOLO26l, YOLO26x, RF-DETR-L, RF-DETR-XL.  
**LCOD/open-vocabulary/grounding:** Grounding DINO, YOLO-World, YOLOE, Grounding DINO 1.5 Edge, LocateAnything-3B.


## Session 1 — Workspace và thư mục quản lý


In [ ]:
from __future__ import annotations
import os, sys
from pathlib import Path
HERE=Path.cwd().resolve(); ROOT=next((p for p in (HERE,HERE/'testmodel') if (p/'benchmark_object_detection.ipynb').exists()),None)
if ROOT is None: raise RuntimeError('Run notebook from project directory or parent')
CACHE=ROOT/'cache'; HF=CACHE/'huggingface'; MODELS=CACHE/'models'; VIDEOS=ROOT/'data'/'videos'; SESSIONS=ROOT/'outputs'/'live_sessions'
for p in (CACHE,HF,MODELS,VIDEOS,SESSIONS): p.mkdir(parents=True,exist_ok=True)
def read_env(path):
    out={}
    if path.exists():
        for raw in path.read_text(encoding='utf-8').splitlines():
            line=raw.strip()
            if line and not line.startswith('#') and '=' in line:
                k,v=line.split('=',1); out[k.strip()]=v.strip().strip('\"').strip("'")
    return out
ENVS=[read_env(ROOT/x) for x in ('.env.local','.env.gdino15-edge','.env.locateanything','.env.main')]
def env_value(k,default=''):
    return os.environ.get(k) or next((e[k] for e in ENVS if e.get(k)),default)
print('Project root:',ROOT)


## Session 2 — Cache và dependency bootstrap


In [ ]:
import importlib.util
os.environ.update({'HF_HOME':str(HF),'HUGGINGFACE_HUB_CACHE':str(HF/'hub'),'TRANSFORMERS_CACHE':str(HF/'hub'),'TORCH_HOME':str(CACHE/'torch'),'ULTRALYTICS_HOME':str(CACHE/'ultralytics')})
required=('torch','torchvision','ultralytics','transformers','huggingface_hub','ipywidgets','cv2','PIL','numpy')
missing=[m for m in required if importlib.util.find_spec(m) is None]
if missing: raise RuntimeError('Missing packages: '+', '.join(missing))
import transformers
if tuple(map(int,transformers.__version__.split('.')[:2])) < (5,1): raise RuntimeError('Main env requires Transformers >=5.1; LocateAnything uses separate env')
print('Transformers:',transformers.__version__)


## Session 3 — Runtime, logging và GPU


In [ ]:
import asyncio, concurrent.futures, csv, json, logging, time
from dataclasses import dataclass
import cv2, ipywidgets as widgets, numpy as np, torch
from IPython.display import display
logging.basicConfig(level=logging.INFO,format='%(asctime)s | %(levelname)s | %(message)s',force=True); LOG=logging.getLogger('lcod-lab')
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'; DTYPE=torch.float16 if DEVICE=='cuda' else torch.float32
VIDEO_SUFFIXES={'.mp4','.avi','.mov','.mkv','.webm'}
print('Device:',DEVICE); print('GPU:',torch.cuda.get_device_name(0) if DEVICE=='cuda' else 'CPU')


## Session 4 — Model registry và cấu trúc kết quả


In [ ]:
CONFIG={
'yolo26l':{'name':'YOLO26l','type':'yolo','weight':MODELS/'yolo26l.pt','prompted':False},
'yolo26x':{'name':'YOLO26x','type':'yolo','weight':MODELS/'yolo26x.pt','prompted':False},
'rfdetr_l':{'name':'RF-DETR-L','type':'rfdetr','variant':'large','prompted':False},
'rfdetr_xl':{'name':'RF-DETR-XL','type':'rfdetr','variant':'xlarge','prompted':False},
'grounding_dino':{'name':'Grounding DINO','type':'grounding_dino','repo':env_value('GROUNDING_DINO_REPO','IDEA-Research/grounding-dino-base'),'prompted':True},
'yolo_world':{'name':'YOLO-World','type':'yolo_world','weight':env_value('YOLO_WORLD_WEIGHT','yolov8l-worldv2.pt'),'prompted':True},
'yoloe':{'name':'YOLOE','type':'yoloe','weight':env_value('YOLOE_WEIGHT','yoloe-26l-seg.pt'),'prompted':True},
'gdino15_edge':{'name':'Grounding DINO 1.5 Edge','type':'gdino15_edge','prompted':True,'remote':True},
'locateanything_3b':{'name':'LocateAnything-3B','type':'locateanything','model':env_value('LOCATEANYTHING_MODEL','nvidia/LocateAnything-3B'),'prompted':True}}
@dataclass
class Det: label:str; score:float; x1:float; y1:float; x2:float; y2:float
@dataclass
class Result: detections:list[Det]; ms:float


## Session 5 — Video và visualization utilities


In [ ]:
def videos(): return sorted(p for p in VIDEOS.rglob('*') if p.suffix.lower() in VIDEO_SUFFIXES)
def video_text(path):
    c=cv2.VideoCapture(str(path)); fps=c.get(cv2.CAP_PROP_FPS) or 0; n=int(c.get(cv2.CAP_PROP_FRAME_COUNT) or 0); w=int(c.get(cv2.CAP_PROP_FRAME_WIDTH) or 0); h=int(c.get(cv2.CAP_PROP_FRAME_HEIGHT) or 0); c.release(); return f'{path.name} | {w}x{h} | {n/fps if fps else 0:.1f}s'
def paint(frame,dets):
    out=frame.copy()
    for d in dets:
        x1,y1,x2,y2=map(int,(d.x1,d.y1,d.x2,d.y2)); cv2.rectangle(out,(x1,y1),(x2,y2),(45,220,120),2); score='' if np.isnan(d.score) else f' {d.score:.2f}'; cv2.putText(out,f'{d.label}{score}',(x1,max(18,y1-6)),cv2.FONT_HERSHEY_SIMPLEX,.5,(45,220,120),2)
    return out


## Session 6 — Model adapters


In [ ]:
from lcod_adapters import Adapter, YOLOAdapter, RFDETRAdapter, GroundingDinoAdapter, YOLOWorldAdapter, YOLOEAdapter, GroundingDino15EdgeAdapter, LocateAnythingAdapter
CTX={'ROOT':ROOT,'HF':HF,'DEVICE':DEVICE,'Det':Det,'Result':Result,'env_value':env_value}


## Session 7 — Adapter registry và model manager


In [ ]:
ADAPTERS={'yolo26l':YOLOAdapter,'yolo26x':YOLOAdapter,'rfdetr_l':RFDETRAdapter,'rfdetr_xl':RFDETRAdapter,'grounding_dino':GroundingDinoAdapter,'yolo_world':YOLOWorldAdapter,'yoloe':YOLOEAdapter,'gdino15_edge':GroundingDino15EdgeAdapter,'locateanything_3b':LocateAnythingAdapter}
class Manager:
    def __init__(self): self.key=None; self.a=None
    def load(self,key,progress=None):
        if self.key==key and self.a is not None: return
        self.unload(); self.a=ADAPTERS[key](key,CONFIG,CTX); self.a.prepare(progress or Progress()); self.key=key
    def predict(self,key,frame,conf,prompt=''): self.load(key); return self.a.predict(frame,conf,prompt)
    def unload(self):
        if self.a is not None: self.a.unload()
        self.key=self.a=None
MANAGER=Manager()


## Session 8 — Model preflight + smoke test


In [ ]:
class Progress:
    def __init__(self): self.bar=widgets.IntProgress(value=0,min=0,max=100,description='Overall'); self.text=widgets.HTML('Waiting.'); self.view=widgets.VBox([widgets.HTML('<b>Preparing models</b>'),self.bar,self.text])
    def note(self,text): self.text.value=f'<code>{text}</code>'
    def task(self,_percent,text): self.note(text)
def smoke_frame():
    f=np.full((640,640,3),128,dtype=np.uint8); cv2.rectangle(f,(120,140),(300,430),(210,210,210),-1); return f
def prepare_all(progress):
    report={}; test=smoke_frame(); total=len(CONFIG)
    for i,key in enumerate(CONFIG,1):
        progress.bar.value=int((i-1)*100/total); progress.note(f'[{i}/{total}] {CONFIG[key]["name"]}')
        try:
            MANAGER.load(key,progress); r=MANAGER.predict(key,test,.25,'person' if CONFIG[key].get('prompted') else ''); report[key]={'state':'ready','detail':'Load + smoke inference verified','smoke_ms':round(r.ms,3)}
        except Exception as e: report[key]={'state':'failed','detail':f'{type(e).__name__}: {e}'}
        finally: MANAGER.unload()
    progress.bar.value=100; (CACHE/'model_preflight.json').write_text(json.dumps({'models':report},indent=2),encoding='utf-8'); return report


## Session 9 — Ready-to-play UI


In [ ]:
EXECUTOR=concurrent.futures.ThreadPoolExecutor(max_workers=1)
class Lab:
    def __init__(self,ready,report):
        self.model=widgets.Dropdown(options=[(CONFIG[k]['name'],k) for k in ready],description='Model:',layout=widgets.Layout(width='420px')); self.prompt=widgets.Text(value='person',description='Prompt:',layout=widgets.Layout(width='760px')); self.source=widgets.ToggleButtons(options=[('Video file','video'),('Live webcam','camera')],value='camera',description='Input:'); self.video=widgets.Dropdown(options=[(video_text(p),str(p)) for p in videos()],description='Video:',layout=widgets.Layout(width='760px')); self.camera=widgets.BoundedIntText(value=0,min=0,max=9,description='Camera #:'); self.play=widgets.Button(description='Play',button_style='success'); self.stop=widgets.Button(description='Stop',button_style='warning',disabled=True); self.conf=widgets.FloatSlider(value=.35,min=.05,max=.95,step=.05,description='Confidence'); self.image=widgets.Image(format='jpeg',layout=widgets.Layout(width='760px')); self.status=widgets.HTML(); self.metrics=widgets.HTML(); self.running=False; self.rows=[]; self.capture=None
        self.model.observe(self.change_model,names='value'); self.play.on_click(lambda _:asyncio.create_task(self.run())); self.stop.on_click(lambda _:self.stop_session()); self.change_model({'new':self.model.value}); failed='<br>'.join(f'<b>{CONFIG[k]["name"]}</b>: {v["detail"]}' for k,v in report.items() if v['state']!='ready') or 'None'; self.view=widgets.VBox([widgets.HTML('<h3>Interactive Object Detection + LCOD Benchmark</h3>'),self.model,self.prompt,self.source,self.video,self.camera,widgets.HBox([self.play,self.stop]),self.conf,widgets.HTML('<b>Unavailable after preflight</b><br>'+failed),self.status,self.metrics,self.image])
    def show(self): display(self.view)
    def change_model(self,c): self.prompt.disabled=not CONFIG[c.get('new')].get('prompted') if c.get('new') else True
    def stop_session(self): self.running=False; self.stop.disabled=True; self.capture.release() if self.capture is not None else None
    async def run(self):
        if self.running or not self.model.value:return
        key=self.model.value; query=self.prompt.value.strip() if CONFIG[key].get('prompted') else ''
        if CONFIG[key].get('prompted') and not query: self.status.value='<b>Prompt required.</b>'; return
        self.running=True; self.play.disabled=True; self.stop.disabled=False; loop=asyncio.get_running_loop(); self.rows=[]
        try:
            await loop.run_in_executor(EXECUTOR,MANAGER.load,key); self.capture=cv2.VideoCapture(int(self.camera.value)) if self.source.value=='camera' else cv2.VideoCapture(str(self.video.value)); source=f'webcam_{self.camera.value}' if self.source.value=='camera' else Path(self.video.value).stem
            while self.running:
                ok,frame=self.capture.read()
                if not ok: break
                r=await loop.run_in_executor(EXECUTOR,MANAGER.predict,key,frame,self.conf.value,query); painted=paint(frame,r.detections); ok2,jpg=cv2.imencode('.jpg',painted); self.image.value=jpg.tobytes() if ok2 else b''; self.rows.append({'frame':len(self.rows)+1,'objects':len(r.detections),'inference_ms':round(r.ms,3),'model':key,'prompt':query,'input':source}); self.metrics.value=f'<b>Objects:</b> {len(r.detections)} | <b>Inference:</b> {r.ms:.1f} ms'
            if self.rows:
                out=SESSIONS/f'{int(time.time())}_{key}_{source}.csv'; f=out.open('w',newline='',encoding='utf-8'); w=csv.DictWriter(f,fieldnames=self.rows[0].keys()); w.writeheader(); w.writerows(self.rows); f.close(); self.status.value=f'<b>Metrics:</b> {out}'
        except Exception as e: self.status.value=f'<b style="color:#b91c1c">{type(e).__name__}: {e}</b>'
        finally:
            if self.capture is not None:self.capture.release()
            self.capture=None; self.running=False; self.play.disabled=False; self.stop.disabled=True


## Session 10 — Launch gate


In [ ]:
PROGRESS=Progress(); display(PROGRESS.view); REPORT=prepare_all(PROGRESS); READY_KEYS=[k for k in CONFIG if REPORT[k]['state']=='ready']
if env_value('VLM_REQUIRE_ALL_MODELS_READY','0')=='1' and len(READY_KEYS)!=len(CONFIG): raise RuntimeError('Strict preflight failed')
if not READY_KEYS: raise RuntimeError('No model passed preflight')
LAB=Lab(READY_KEYS,REPORT); LAB.show()


## Session 11 — Cách sử dụng

1. Chạy **Run All**.
2. Chọn model và input.
3. Với 5 model LCOD/open-vocabulary, nhập prompt như `bottle`, `red bottle`, `red bottle next to the laptop`.
4. Bấm **Play**.

`Grounding DINO 1.5 Edge` là remote API; latency của nó bao gồm network/service time. `LocateAnything-3B` dùng interpreter riêng khai báo trong `.env.locateanything`.
